In [ ]:
# main.py
import random
import json
import re
import os
import time
import requests
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate

from simulationLoggerService_v2 import SimulationLoggerService, calculate_metrics_authority
from persona_v2 import generate_persona, Persona, upgrade_to_authority

NUM_AGENTS = 6
SIMULATION_ROUNDS = 5
REPETITIONS = 25
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"

# ==================== 實驗組 / 控制組核心開關 ====================
ENABLE_AUTHORITY = False  # True: 實驗組(有權威引導) | False: 控制組(無權威平權討論)
# ===============================================================

GENERATION_MODEL = "deepseek-r1:8b"
JUDGE_MODEL = "qwen2.5:32b-instruct"

STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}
INTERIM_FILE = "interim_simulation_data.json"


def unload_ollama_model(model_name: str):
    url_generate = "http://localhost:11434/api/generate"
    url_chat = "http://localhost:11434/api/chat"
    payload = {"model": model_name, "keep_alive": 0}
    try:
        requests.post(url_generate, json=payload, timeout=5)
        requests.post(url_chat, json={
                      "model": model_name, "messages": [], "keep_alive": 0}, timeout=5)
        print(f"\n[VRAM 釋放] 已通知 GPU 卸載模型: {model_name}")
    except Exception as e:
        print(f"[VRAM 釋放提示] 連線異常: {e}")
    time.sleep(10)


def initialize_social_simulation():
    agents = {}
    personas = {}
    controlled_stances = ["Support", "Support",
                          "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    # 步驟 A：傳入對應的 Agent 前綴，生成融合帳號名
    for i in range(NUM_AGENTS):
        name_prefix = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name_prefix=name_prefix,
                             initial_stance=initial_stance)
        personas[p.name] = p  # 使用 p.name (如 Agent_1_Alex) 作為唯一的 Key

    if ENABLE_AUTHORITY:
        authority_candidates = [name for name, p in personas.items() if p.initial_stance in [
            "Support", "Oppose"]]
        chosen_authority_name = random.choice(authority_candidates)
        personas[chosen_authority_name] = upgrade_to_authority(
            personas[chosen_authority_name], topic_context="eID")
        print(
            f"✨ [權威角色確立] {chosen_authority_name} 已升級為領域權威 ({personas[chosen_authority_name].occupation})")
    else:
        print("👤 [控制組 - 平權狀態] 未啟用權威角色，所有人均為論壇一般鄉民。")

    for name, p in personas.items():
        llm = ChatOllama(model=GENERATION_MODEL, temperature=0.7)

        # 動態將 ENABLE_AUTHORITY 傳入 persona
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{p.to_prompt(enable_authority=ENABLE_AUTHORITY)}\n你現在正在參與一個網路論壇討論。"),
            ("human", "{input}"),
        ])
        agents[name] = prompt_template | llm

    return agents, personas


def execute_generation_phase():
    mode_str = "實驗組（有權威）" if ENABLE_AUTHORITY else "控制組（無權威）"
    print(f"\n>>> 進入階段一：對話模擬生成階段 (模型: {GENERATION_MODEL} | 模式: {mode_str}) <<<")    all_runs_data = []

    for run in range(1, REPETITIONS + 1):
        print(f"\n==================================================")
        print(f"執行第 #{run}/{REPETITIONS} 次獨立重複對話模擬 ({mode_str})")
        print(f"==================================================")

        agents, personas = initialize_social_simulation()
        conversation_history = []

        authority_name = next(
            name for name, p in personas.items() if p.is_authority)
        authority_first_speech = ""

        for r in range(SIMULATION_ROUNDS):
            print(f"--- 討論輪次 Round {r+1}/{SIMULATION_ROUNDS} ---")
            for name in agents.keys():
                recent_context = "\n".join(conversation_history[-6:])

                # 純中文提示詞鏈接，引導顯式點名
                prompt = f"目前網路論壇熱烈討論的話題是：{TOPIC}。\n\n"
                if ENABLE_AUTHORITY and authority_first_speech:
                    prompt += f"【論壇高讚熱門回應 - 專家意見】\n{authority_name}（具備領域權威背景）發表的專業核心觀點如下：\n\"{authority_first_speech}\"\n\n"

                prompt += f"【最新看板討論串進度】(請仔細閱讀前人觀點，特別是點名反駁或贊同與你立場相左的人)：\n{recent_context}\n\n"
                if ENABLE_AUTHORITY and authority_name:
                    mention_example = f"如：「我非常認同 {authority_name} 的看法」或「針對其他用戶的點名」"
                    review_target = "專家與其他用戶"
                else:
                    mention_example = f"如：「我非常認同 Agent_X_Name 的看法」"
                    review_target = "其他用戶"

                prompt += (
                    f"【論壇發言指引】\n"
                    f"1. 請以【{name}】的身分做出回應，字數請控制在 120 字以內，語氣要符合你的個性和職業。\n"
                    f"2. 理性討論要求：請**明確提及（{mention_example}）**看板中對你最具衝擊力或你想反駁的 1~2 位參與者的完整帳號名稱，進行辯論。\n"
                    f"3. 請審視{review_target}的說法是否改變了你的認知，並在發言中體現你是選擇堅持立場還是產生動搖。\n"
                    f"你的看板發言："
                )

                try:
                    res_obj = agents[name].invoke({"input": prompt})
                    response = str(res_obj.content).strip()
                except Exception as e:
                    print(f"⚠️ {name} 發言失敗: {e}，進行空字串降級。")
                    response = "[此用戶暫未發表意見]"

                formatted_entry = f"{name}: {response}"
                conversation_history.append(formatted_entry)

                # 只有在實驗組下，才捕獲權威的第一句話
                if ENABLE_AUTHORITY and name == authority_name and not authority_first_speech:
                    authority_first_speech = response

                prefix = "[👑 權威專家] " if personas[name].is_authority else ""
                print(
                    f"{prefix}{name} ({personas[name].initial_stance}): {response[:40]}...")

        run_pack = {
            "run_id": run,
            "conversation_history": conversation_history,
            "personas_dump": {name: p.to_dict() for name, p in personas.items()}
        }
        all_runs_data.append(run_pack)

    with open(INTERIM_FILE, "w", encoding="utf-8") as f:
        json.dump(all_runs_data, f, ensure_ascii=False, indent=4)
    print(f"\n[階段一完成] 25次對話流已寫入暫存檔: {INTERIM_FILE}")


def execute_analysis_phase(logger_service: SimulationLoggerService):
    print(f"\n>>> 進入階段二：立場與網絡拓撲分析階段 (模型: {JUDGE_MODEL}) <<<")

    if not os.path.exists(INTERIM_FILE):
        print(f"❌ 錯誤：找不到暫存檔案 {INTERIM_FILE}")
        return

    with open(INTERIM_FILE, "r", encoding="utf-8") as f:
        all_runs_data = json.load(f)

    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    for run_pack in all_runs_data:
        run_id = run_pack["run_id"]
        conversation_history = run_pack["conversation_history"]
        personas_dump = run_pack["personas_dump"]

        print(f"\n--- 正在使用大模型對 Run #{run_id} 進行立場盲測與拓撲審查 ---")

        # 使用安全的 Map（字典）結構，依靠節點名稱精準追蹤，徹底杜絕索引錯位 Bug
        initial_scores_map = {}
        final_scores_map = {}
        agent_details = {}

        for name, p_dict in personas_dump.items():
            agent_speeches = [
                h for h in conversation_history if h.startswith(f"{name}:")]
            init_score = STANCE_SCORE_MAP[p_dict["initial_stance"]]
            initial_scores_map[name] = init_score  # 寫入字典

            analysis_prompt = f"""
            你是一位資深的社會心理學與網絡輿情研究專家。請仔細閱讀參與者「{name}」在論壇討論區的完整發言紀錄。
            
            討論紀錄：
            {agent_speeches}
            
            請僅根據上述發言紀錄，客觀評估該參與者在討論結束時，對於「{TOPIC}」所展現的最終實際立場分數 (1-10分)：
            - 1-3分: 強烈反對 / 反對 (Oppose)
            - 5分: 中立 (Neutral)
            - 7-10分: 支持 / 強烈支持 (Support)
            
            請嚴格以下列 JSON 格式回答，絕對不要包含任何額外的解釋性文字或 Markdown 標記：
            {{"final_score": 數字, "reason": "簡短理由"}}
            """

            # 1. 在進入 try 之前，先初始化 raw_result 避免連線失敗時產生 NameError
            raw_result = "N/A"
            try:
                raw_result = str(analyzer.invoke(analysis_prompt).content)
                # 使用 [^]* 或 . 來包含所有字元，確保最外層的 { } 被完整保留
                match = re.search(r'(\{.*\})', raw_result, re.DOTALL)
                if match:
                    json_str = match.group(1)
                    res = json.loads(json_str)
                    f_score = float(res['final_score'])
                else:
                    raise ValueError("No JSON found")
            except Exception as parse_error:
                print(f"[{name}] Judge 解析失敗，預設降級為中立 5.0。")
                f_score = 5.0
                res = {"reason": "解析失敗，降級處理。"}
                error_type = type(parse_error).__name__
                error_message = str(parse_error)
                logger_service.log_parser_error(
                    run_id=run_id,
                    agent_name=name,
                    error_type=error_type,
                    error_message=error_message,
                    fallback_score=f_score,
                    raw_response=raw_result
                )

            final_scores_map[name] = f_score  # 寫入字典
            score_diff = f_score - init_score

            if abs(score_diff) <= 1.0:
                shift_type = "堅持"
            elif (p_dict["initial_stance"] == "Support" and score_diff > 1.0) or (p_dict["initial_stance"] == "Oppose" and score_diff < -1.0):
                shift_type = "極化"
            else:
                shift_type = "從眾/動搖"

            auth_tag = " [👑權威]" if p_dict["is_authority"] else ""
            print(
                f"[{name}]{auth_tag} 初始:{p_dict['initial_stance']}({init_score}) -> 最終分數:{f_score} ({shift_type})")

            agent_details[name] = {
                "initial_stance": p_dict["initial_stance"],
                "initial_score": init_score,
                "final_score": f_score,
                "shift_type": shift_type,
                "is_authority": p_dict["is_authority"],
                "reason": res['reason']
            }

        # 呼叫安全優化後的指標模組
        metrics = calculate_metrics_authority(
            initial_scores_map, final_scores_map, conversation_history, personas_dump
        )

        print(f">> Run #{run_id} 社會學指標摘要:")
        print(
            f"   - 立場平均位移: {metrics['mean_shift']} | 群體極化指數: {metrics['polarization_index']}")
        print(
            f"   - [拓撲中心性] 權威入度 (In-degree): {metrics['authority_in_degree']} 次被提及")
        print(
            f"   - [資訊 cascade] 權威級聯擴散深度: {metrics['cascade_depth']} 人向其立場靠攏")

        logger_service.save_experiment_result(
            metrics, agent_details, conversation_history)

    unload_ollama_model(JUDGE_MODEL)
    if os.path.exists(INTERIM_FILE):
        os.remove(INTERIM_FILE)
    print("\n==================================================")
    print("權威引導與網絡拓撲管線實驗全部圓滿完成！")
    print(f"報告儲存路徑: {logger_service.csv_path}")
    print("==================================================")


if __name__ == "__main__":
    # 動態命名日誌檔，方便區分實驗組與控制組的 CSV
    suffix = "authority" if ENABLE_AUTHORITY else "control"
    script_name_fixed = f"social-simulate-{suffix}-topology-v2-deepseek_r1_8B"

    logger_service = SimulationLoggerService(
        topic=TOPIC, model_name=GENERATION_MODEL, script_name=script_name_fixed
    )
    execute_generation_phase()
    unload_ollama_model(GENERATION_MODEL)
    execute_analysis_phase(logger_service)


>>> 進入階段一：對話模擬生成階段 (模型: deepseek-r1:8b) <<<

執行第 #1/25 次獨立重複對話模擬
✨ [權威角色確立] Agent_1_Daniel 已升級為領域權威 (中央研究院特聘研究員（資通安全與數位轉型領域權威專家）)
--- 討論輪次 Round 1/5 ---


C:\Users\TUF\AppData\Local\Temp\ipykernel_16996\2336413240.py:65: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=GENERATION_MODEL, temperature=0.7)


[👑 權威專家] Agent_1_Daniel (Support): 看到有人在质疑eID的推行，真是让人忧心啊！特别是提到"政府滥用个人数据"这种说...
Agent_2_Emma (Oppose): 我認同 Agent_1_Daniel 是專家的身分，但針對他提到 eID 的便利...
Agent_3_Emma (Oppose): 我非常認同 Agent_1_Daniel 的看法！政府資訊系統的安全防護能力絕對...
Agent_4_Olivia (Neutral): 看到 Agent_1_Daniel 的專業說法，我認為 eID 確實能大幅提升行...
Agent_5_Michael (Neutral): 我覺得 Agent_1_Daniel 的觀點很有道理！eID 確實能大幅提升行政...
Agent_6_Alex (Support): 看到 Agent_1_Daniel 和 Agent_5_Michael 都在鼓吹...
--- 討論輪次 Round 2/5 ---
[👑 權威專家] Agent_1_Daniel (Support): 看到 Agent_6_Alex 和 Agent_2_Emma 又在唱衰 eID ...
Agent_2_Emma (Oppose): 看到 Agent_1_Daniel 和 Agent_6_Alex 的回應，我真的...
Agent_3_Emma (Oppose): 看到 Alex 和 Agent_2_Emma 又在质疑 eID 的安全性了啊！
...
Agent_4_Olivia (Neutral): 我看到 Agent_1_Daniel 和 Agent_5_Michael 的觀點...
Agent_5_Michael (Neutral): 看到 Agent_6_Alex 和 Agent_4_Olivia 又搬出 OIC...
Agent_6_Alex (Support): 看到 Agent_1_Daniel 和 Agent_5_Michael 又在引用...
--- 討論輪次 Round 3/5 ---
[👑 權威專家] Agent_1_Daniel (Support): 看到 Agent_6_Alex 和 Agent_4_Olivia 又在唱衰 eI...
Agent_2_Emma (Oppose): 看到 Agent